# 12 - Extension to ETTm1: Does Regime-Aware Head Selection Generalize to a Different Dataset?

**Project:** Regime-Conditional Attention Head Selection for Time Series Transformers (ReCAHS)

The proposal states: *"If the regime-specific head importance patterns are clear, the experiments will be extended to other prediction lengths and datasets such as ETTm1 and Weather."* Experiments 03-10 found clear regime-specific head behavior on ETTh1 (e.g. heads with opposite-signed importance across regimes) and a joint/greedy selection method that beat every other pruning criterion tried — so this notebook tests whether that finding transfers to **ETTm1**, the same electricity-transformer sensor recorded at 15-minute instead of hourly resolution (same 7 columns, ~4x more windows for the same real-world time span).

**Scope decisions (to keep this a single runnable notebook instead of re-doing notebooks 01/03-10 from scratch):**
- **Same architecture as B4** (`d_model=128, n_heads=8, e_layers=3, seq_len=336, pred_len=96`) is trained directly on ETTm1, rather than re-running the whole B0-B4 baseline sweep — the question here is whether the *method* transfers, not re-optimizing hyperparameters for a new dataset.
- **STL period is adapted to the new sampling rate**: ETTh1 uses `period=24` (24 hourly steps = 1 day); ETTm1 uses `period=96` (96 fifteen-minute steps = 1 day) so "daily seasonality" means the same real-world thing on both datasets.
- **Only the two strongest methods from the ETTh1 experiments are re-tested**: static 25% pruning (Experiment 04) and joint/greedy dynamic 75% keep (Experiment 10, the best method overall) vs. the unpruned baseline. Random pruning, magnitude-based pruning, independent-scoring dynamic pruning, fine-tuning, and efficiency metrics already answered their respective methodological questions on ETTh1 and are not repeated here — re-running them would mostly re-confirm conclusions already established, at high additional runtime cost.
- **Head-importance leave-one-out scoring and the joint greedy search both use a random subsample of validation windows** (same approach as Experiment 10, `GREEDY_SUBSAMPLE_SIZE=256` per regime) to keep runtime bounded — ETTm1's validation set has roughly 4x more windows than ETTh1's. Final reported accuracy numbers are still evaluated on the **full** validation and test sets.

**Expected runtime: this is by far the longest-running notebook in this project — likely 1.5-3 hours on a T4 GPU**, dominated by training the baseline on ~4x more data than ETTh1. Make sure the Colab runtime will not go idle/disconnect during training (keep the tab active, or use Colab Pro's background execution if available).


## 1. Mount Google Drive and import core libraries

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path
import sys
import os
import shutil
import random
import copy

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from tqdm.auto import tqdm

## 2. Define project paths

In [3]:
PROJECT_DIR = Path(
    "/content/drive/MyDrive/BIL401_Regime_Head_Pruning"
)

CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
ETTM1_DIR = PROJECT_DIR / "ettm1_extension"
ETTM1_REGIME_DIR = ETTM1_DIR / "regime_detection"
ETTM1_RESULTS_DIR = ETTM1_DIR / "results"

for d in [ETTM1_DIR, ETTM1_REGIME_DIR, ETTM1_RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("ETTM1_DIR:", ETTM1_DIR)

PROJECT_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning
ETTM1_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/ettm1_extension


## 3. Set up Time-Series-Library

In [4]:
TSLIB_DIR = Path("/content/Time-Series-Library")

if not TSLIB_DIR.exists():
    %cd /content
    !git clone https://github.com/thuml/Time-Series-Library.git
else:
    print("Time-Series-Library already exists:", TSLIB_DIR)

sys.path.insert(0, str(TSLIB_DIR))

/content
Cloning into 'Time-Series-Library'...
remote: Enumerating objects: 2295, done.
remote: Total 2295 (delta 0), reused 0 (delta 0), pack-reused 2295 (from 1)
Receiving objects: 100% (2295/2295), 78.43 MiB | 18.41 MiB/s, done.
Resolving deltas: 100% (1571/1571), done.


In [5]:
%cd /content/Time-Series-Library
!pip install -q patool sktime scikit-base statsmodels --no-deps
!pip install -q --no-deps einops
!pip install reformer-pytorch --no-deps
!pip install local-attention --no-deps
!pip install hyper_connections --no-deps
!pip install axial_positional_embedding --no-deps
!pip install product_key_memory --no-deps
!pip install colt5_attention --no-deps

/content/Time-Series-Library
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.5/37.5 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 19.3 MB/s eta 0:00:00


## 4. Download the ETTm1 dataset

In [6]:
tslib_data_path = TSLIB_DIR / "dataset/ETDataset/ETT-small/ETTm1.csv"
tslib_data_path.parent.mkdir(parents=True, exist_ok=True)

drive_data_path = PROJECT_DIR / "data" / "ETTm1.csv"

if drive_data_path.exists():
    shutil.copy2(drive_data_path, tslib_data_path)
    print("ETTm1 copied from Drive.")
elif not tslib_data_path.exists():
    print("Downloading ETTm1 from GitHub...")
    !wget -q https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTm1.csv -O "{tslib_data_path}"
else:
    print("ETTm1 already present.")

df = pd.read_csv(tslib_data_path)
print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (69680, 8)


,date,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
0,2016-07-01 00:00:00,5.827,2.009,1.599,0.462,4.203,1.340,30.531000
1,2016-07-01 00:15:00,5.760,2.076,1.492,0.426,4.264,1.401,30.459999
2,2016-07-01 00:30:00,5.760,1.942,1.492,0.391,4.234,1.310,30.038000
3,2016-07-01 00:45:00,5.760,1.942,1.492,0.426,4.234,1.310,27.013000
4,2016-07-01 01:00:00,5.693,2.076,1.492,0.426,4.142,1.371,27.787001


## 5. Train the B4-equivalent baseline on ETTm1

Same hyperparameters as the B4 baseline on ETTh1, just pointed at ETTm1 (`--data ETTm1 --freq t`, minute-level time features). With ~4x more training windows than ETTh1, this will take noticeably longer per epoch.

In [7]:
%cd /content/Time-Series-Library

!python -u run.py \
  --task_name long_term_forecast \
  --is_training 1 \
  --root_path ./dataset/ETDataset/ETT-small/ \
  --data_path ETTm1.csv \
  --model_id ETTm1_336_96_dm128_h8 \
  --model PatchTST \
  --data ETTm1 \
  --features M \
  --freq t \
  --seq_len 336 \
  --label_len 48 \
  --pred_len 96 \
  --enc_in 7 \
  --dec_in 7 \
  --c_out 7 \
  --e_layers 3 \
  --d_layers 1 \
  --factor 3 \
  --d_model 128 \
  --d_ff 256 \
  --n_heads 8 \
  --batch_size 32 \
  --train_epochs 10 \
  --patience 3 \
  --learning_rate 0.0001 \
  --des ettm1_extension \
  --itr 1

/content/Time-Series-Library
Using GPU
Args in experiment:
Basic Config
  Task Name:          long_term_forecast  Is Training:        1                   
  Model ID:           ETTm1_336_96_dm128_h8Model:              PatchTST            

Data Loader
  Data:               ETTm1               Root Path:          ./dataset/ETDataset/ETT-small/
  Data Path:          ETTm1.csv           Features:           M                   
  Target:             OT                  Freq:               t                   
  Checkpoints:        ./checkpoints/      

Forecasting Task
  Seq Len:            336                 Label Len:          48                  
  Pred Len:           96                  Seasonal Patterns:  Monthly             
  Inverse:            0                   

Model Parameters
  Top k:              5                   Num Kernels:        6                   
  Enc In:             7                   Dec In:             7                   
  C Out:              7            

## 6. Locate the checkpoint and copy it to Drive

In [8]:
checkpoint_root = Path("/content/Time-Series-Library/checkpoints")

ettm1_matches = [p for p in checkpoint_root.iterdir() if "ETTm1_336_96_dm128_h8" in p.name]

print("Found checkpoint folders:")
for p in ettm1_matches:
    print(p)

if len(ettm1_matches) != 1:
    raise RuntimeError(f"1 ETTm1 checkpoint klasörü bekleniyordu, {len(ettm1_matches)} bulundu.")

source_ckpt = ettm1_matches[0]
target_ckpt = CHECKPOINT_DIR / "ETTm1_B4_equivalent"

if target_ckpt.exists():
    shutil.rmtree(target_ckpt)

shutil.copytree(source_ckpt, target_ckpt)
print("\nETTm1 checkpoint saved to:", target_ckpt)

Found checkpoint folders:
/content/Time-Series-Library/checkpoints/long_term_forecast_ETTm1_336_96_dm128_h8_PatchTST_ETTm1_ftM_sl336_ll48_pl96_dm128_nh8_el3_dl1_df256_expand2_dc4_fc3_ebtimeF_dtTrue_ettm1_extension_0

ETTm1 checkpoint saved to: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/checkpoints/ETTm1_B4_equivalent


## 7. Reconstruct the model arguments for evaluation

In [9]:
from argparse import Namespace

args = Namespace(
    task_name="long_term_forecast",
    is_training=0,
    model_id="ETTm1_336_96_dm128_h8",
    model="PatchTST",

    data="ETTm1",
    root_path="./dataset/ETDataset/ETT-small/",
    data_path="ETTm1.csv",
    features="M",
    target="OT",
    freq="t",
    checkpoints="./checkpoints/",

    seq_len=336,
    label_len=48,
    pred_len=96,
    seasonal_patterns="Monthly",
    inverse=False,

    enc_in=7,
    dec_in=7,
    c_out=7,
    d_model=128,
    n_heads=8,
    e_layers=3,
    d_layers=1,
    d_ff=256,
    moving_avg=25,
    factor=3,
    distil=True,
    dropout=0.1,
    embed="timeF",
    activation="gelu",
    output_attention=False,

    patch_len=16,
    stride=8,
    padding_patch="end",
    revin=1,
    affine=0,
    subtract_last=0,
    decomposition=0,
    kernel_size=25,
    individual=0,

    num_workers=0,
    itr=1,
    train_epochs=10,
    batch_size=32,
    patience=3,
    learning_rate=0.0001,
    des="ettm1_extension",
    loss="MSE",
    lradj="type1",
    use_amp=False,
    augmentation_ratio=0.0,

    use_gpu=torch.cuda.is_available(),
    gpu=0,
    use_multi_gpu=False,
    devices="0",
    gpu_type="cuda",
    expand=2,
    d_conv=4,
    top_k=5,
    num_kernels=6,
    channel_independence=0,
    decomp_method="moving_avg",
    use_norm=1,
    down_sampling_layers=0,
    down_sampling_window=1,
    down_sampling_method=None,
    seg_len=48,

    p_hidden_dims=[128, 128],
    p_hidden_layers=2,
)

print(args)

Namespace(task_name='long_term_forecast', is_training=0, model_id='ETTm1_336_96_dm128_h8', model='PatchTST', data='ETTm1', root_path='./dataset/ETDataset/ETT-small/', data_path='ETTm1.csv', features='M', target='OT', freq='t', checkpoints='./checkpoints/', seq_len=336, label_len=48, pred_len=96, seasonal_patterns='Monthly', inverse=False, enc_in=7, dec_in=7, c_out=7, d_model=128, n_heads=8, e_layers=3, d_layers=1, d_ff=256, moving_avg=25, factor=3, distil=True, dropout=0.1, embed='timeF', activation='gelu', output_attention=False, patch_len=16, stride=8, padding_patch='end', revin=1, affine=0, subtract_last=0, decomposition=0, kernel_size=25, individual=0, num_workers=0, itr=1, train_epochs=10, batch_size=32, patience=3, learning_rate=0.0001, des='ettm1_extension', loss='MSE', lradj='type1', use_amp=False, augmentation_ratio=0.0, use_gpu=True, gpu=0, use_multi_gpu=False, devices='0', gpu_type='cuda', expand=2, d_conv=4, top_k=5, num_kernels=6, channel_independence=0, decomp_method='mov

## 8. Load the model and checkpoint weights

In [10]:
from exp.exp_long_term_forecasting import Exp_Long_Term_Forecast

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

exp = Exp_Long_Term_Forecast(args)
model = exp.model.to(device)

checkpoint = torch.load(target_ckpt / "checkpoint.pth", map_location=device)
model.load_state_dict(checkpoint)
model.eval()

print("ETTm1 baseline model loaded successfully.")

Use GPU: cuda:0
🚀 Lazy Loading: PatchTST ...
ETTm1 baseline model loaded successfully.


## 9. Build the validation and test loaders

In [11]:
from data_provider.data_factory import data_provider

vali_data, vali_loader = data_provider(args, flag="val")
test_data, test_loader = data_provider(args, flag="test")

print("Validation dataset length:", len(vali_data), "| batches:", len(vali_loader))
print("Test dataset length:", len(test_data), "| batches:", len(test_loader))

val 11425
test 11425
Validation dataset length: 11425 | batches: 358
Test dataset length: 11425 | batches: 358


## 10. STL-based regime detection for ETTm1 (validation and test)

Same method as `regime_detection.ipynb` / notebook 07/10, but with `period=96` instead of `period=24` — at 15-minute resolution, one day is 96 steps rather than 24. Split boundaries follow `Time-Series-Library`'s `Dataset_ETT_minute` convention (all sizes scaled by 4x relative to ETTh1's hourly convention).

In [12]:
from statsmodels.tsa.seasonal import STL

SEQ_LEN = 336
PRED_LEN = 96
STL_PERIOD = 96
TARGET_COL = "OT"

TRAIN_SIZE = 12 * 30 * 24 * 4
VAL_SIZE = 4 * 30 * 24 * 4
TEST_SIZE = 4 * 30 * 24 * 4


def safe_variance(values):
    values = np.asarray(values, dtype=np.float64)
    return float(np.var(values)) if len(values) else 0.0


def label_window_with_stl(series_window, period=96):
    result = STL(series_window, period=period, robust=True).fit()

    trend_var = safe_variance(result.trend)
    seasonal_var = safe_variance(result.seasonal)
    residual_var = safe_variance(result.resid)
    total_var = trend_var + seasonal_var + residual_var

    if total_var <= 1e-12:
        trend_score = seasonal_score = residual_score = 0.0
    else:
        trend_score = trend_var / total_var
        seasonal_score = seasonal_var / total_var
        residual_score = residual_var / total_var

    scores = {"trend": trend_score, "seasonal": seasonal_score, "residual": residual_score}
    regime = max(scores, key=scores.get)
    sorted_scores = sorted(scores.values(), reverse=True)
    confidence_margin = sorted_scores[0] - sorted_scores[1]

    return {
        "regime": regime,
        "trend_score": trend_score,
        "seasonal_score": seasonal_score,
        "residual_score": residual_score,
        "confidence_margin": confidence_margin,
    }


def build_regime_df(segment, num_windows, desc):
    target_values = segment[TARGET_COL].to_numpy(dtype=np.float64)
    records = []
    for window_id in tqdm(range(num_windows), desc=desc):
        window = target_values[window_id:window_id + SEQ_LEN]
        label_info = label_window_with_stl(window, period=STL_PERIOD)
        records.append({"window_id": window_id, **label_info})
    result_df = pd.DataFrame(records)
    result_df["is_confident"] = result_df["confidence_margin"] >= 0.05
    return result_df


val_border1 = TRAIN_SIZE - SEQ_LEN
val_border2 = TRAIN_SIZE + VAL_SIZE
validation_segment = df.iloc[val_border1:val_border2].reset_index(drop=True)
num_val_windows = len(validation_segment) - SEQ_LEN - PRED_LEN + 1

test_border1 = TRAIN_SIZE + VAL_SIZE - SEQ_LEN
test_border2 = TRAIN_SIZE + VAL_SIZE + TEST_SIZE
test_segment = df.iloc[test_border1:test_border2].reset_index(drop=True)
num_test_windows = len(test_segment) - SEQ_LEN - PRED_LEN + 1

print("Validation windows:", num_val_windows, "| Test windows:", num_test_windows)

Validation windows: 11425 | Test windows: 11425


In [13]:
val_regime_path = ETTM1_REGIME_DIR / "ettm1_validation_regimes_ot_seq336.csv"
test_regime_path = ETTM1_REGIME_DIR / "ettm1_test_regimes_ot_seq336.csv"

if val_regime_path.exists():
    regime_df = pd.read_csv(val_regime_path)
    print("Loaded existing validation regime labels.")
else:
    regime_df = build_regime_df(validation_segment, num_val_windows, "STL validation regime labeling")
    regime_df.to_csv(val_regime_path, index=False)

if test_regime_path.exists():
    test_regime_df = pd.read_csv(test_regime_path)
    print("Loaded existing test regime labels.")
else:
    test_regime_df = build_regime_df(test_segment, num_test_windows, "STL test regime labeling")
    test_regime_df.to_csv(test_regime_path, index=False)

assert len(vali_data) == len(regime_df), (len(vali_data), len(regime_df))
assert len(test_data) == len(test_regime_df), (len(test_data), len(test_regime_df))

print("\nValidation regime distribution:")
display(regime_df["regime"].value_counts(normalize=True) * 100)
print("\nTest regime distribution:")
display(test_regime_df["regime"].value_counts(normalize=True) * 100)

STL validation regime labeling:   0%|          | 0/11425 [00:00<?, ?it/s]

STL test regime labeling:   0%|          | 0/11425 [00:00<?, ?it/s]


Validation regime distribution:


,proportion
regime,
seasonal,51.273523
trend,30.363239
residual,18.363239



Test regime distribution:


,proportion
regime,
seasonal,46.984683
trend,33.251641
residual,19.763676


## 11. HeadMaskController and evaluation helpers

Same pattern as notebooks 04-10 (mask supports both a single global 2D mask and a per-window 3D batch mask, with PatchTST's channel-independent batch expansion handled via `repeat_interleave`).

In [14]:
class DynamicHeadMaskController:
    def __init__(self, model):
        self.model = model
        self.original_forwards = {}
        self.current_mask = None

    def install(self):
        for layer_idx, encoder_layer in enumerate(self.model.encoder.attn_layers):
            attention_layer = encoder_layer.attention
            if layer_idx in self.original_forwards:
                continue
            self.original_forwards[layer_idx] = attention_layer.forward

            def make_masked_forward(layer_idx, attention_layer):
                def masked_forward(queries, keys, values, attn_mask, tau=None, delta=None):
                    B, L, _ = queries.shape
                    _, S, _ = keys.shape
                    H = attention_layer.n_heads

                    q = attention_layer.query_projection(queries).view(B, L, H, -1)
                    k = attention_layer.key_projection(keys).view(B, S, H, -1)
                    v = attention_layer.value_projection(values).view(B, S, H, -1)

                    out, attn = attention_layer.inner_attention(q, k, v, attn_mask, tau=tau, delta=delta)

                    if self.current_mask is not None:
                        mask = self.current_mask.to(out.device)
                        if mask.ndim == 2:
                            layer_mask = mask[layer_idx].view(1, 1, H, 1)
                        elif mask.ndim == 3:
                            mask_batch = mask.shape[0]
                            if mask_batch != B:
                                if B % mask_batch != 0:
                                    raise ValueError(f"Cannot expand mask batch {mask_batch} to encoder batch {B}.")
                                mask = mask.repeat_interleave(B // mask_batch, dim=0)
                            layer_mask = mask[:, layer_idx, :].view(B, 1, H, 1)
                        else:
                            raise ValueError(f"Unsupported mask shape: {mask.shape}")
                        out = out * layer_mask

                    out = out.view(B, L, -1)
                    return attention_layer.out_projection(out), attn

                return masked_forward

            attention_layer.forward = make_masked_forward(layer_idx, attention_layer)

    def set_mask(self, mask):
        self.current_mask = mask.clone().float()


mse_criterion = nn.MSELoss(reduction="none")
mae_criterion = nn.L1Loss(reduction="none")


def build_batch_dynamic_mask(window_ids, label_df, regime_masks):
    batch_masks = [regime_masks[label_df.iloc[int(w)]["regime"]] for w in window_ids]
    return torch.stack(batch_masks, dim=0)


def compute_with_global_mask(model, loader, label_df, mask_controller, mask, device, pred_len, desc="eval"):
    model.eval()
    mask_controller.set_mask(mask)
    all_records = []
    global_index = 0
    with torch.no_grad():
        for batch in tqdm(loader, desc=desc):
            bx, by, bxm, bym = [t.float().to(device) for t in batch]
            outputs = model(bx, bxm, by, bym)
            true = by[:, -pred_len:, :]
            mse_s = mse_criterion(outputs, true).mean(dim=(1, 2))
            mae_s = mae_criterion(outputs, true).mean(dim=(1, 2))
            for i in range(bx.shape[0]):
                wid = global_index + i
                all_records.append({"window_id": wid, "regime": label_df.iloc[wid]["regime"],
                                     "mse": float(mse_s[i]), "mae": float(mae_s[i])})
            global_index += bx.shape[0]
    result_df = pd.DataFrame(all_records)
    regime_summary = result_df.groupby("regime").agg(mse=("mse", "mean"), mae=("mae", "mean"), count=("window_id", "count")).reset_index()
    overall = {"overall_mse": float(result_df["mse"].mean()), "overall_mae": float(result_df["mae"].mean())}
    return {"overall": overall, "regime_summary": regime_summary, "window_losses": result_df}


def compute_dynamic(model, loader, label_df, regime_masks, mask_controller, device, pred_len, desc="dynamic eval"):
    model.eval()
    all_records = []
    global_index = 0
    with torch.no_grad():
        for batch in tqdm(loader, desc=desc):
            bx, by, bxm, bym = [t.float().to(device) for t in batch]
            window_ids = list(range(global_index, global_index + bx.shape[0]))
            mask_controller.set_mask(build_batch_dynamic_mask(window_ids, label_df, regime_masks))
            outputs = model(bx, bxm, by, bym)
            true = by[:, -pred_len:, :]
            mse_s = mse_criterion(outputs, true).mean(dim=(1, 2))
            mae_s = mae_criterion(outputs, true).mean(dim=(1, 2))
            for i in range(bx.shape[0]):
                wid = global_index + i
                all_records.append({"window_id": wid, "regime": label_df.iloc[wid]["regime"],
                                     "mse": float(mse_s[i]), "mae": float(mae_s[i])})
            global_index += bx.shape[0]
    result_df = pd.DataFrame(all_records)
    regime_summary = result_df.groupby("regime").agg(mse=("mse", "mean"), mae=("mae", "mean"), count=("window_id", "count")).reset_index()
    overall = {"overall_mse": float(result_df["mse"].mean()), "overall_mae": float(result_df["mae"].mean())}
    return {"overall": overall, "regime_summary": regime_summary, "window_losses": result_df}


mask_controller = DynamicHeadMaskController(model)
mask_controller.install()

num_layers = len(model.encoder.attn_layers)
num_heads = model.encoder.attn_layers[0].attention.n_heads
total_heads = num_layers * num_heads
baseline_mask = torch.ones(num_layers, num_heads)

print("Total heads:", total_heads)

Total heads: 24


## 12. Baseline (no pruning) reference on the full validation and test sets

In [15]:
baseline_val = compute_with_global_mask(model, vali_loader, regime_df, mask_controller, baseline_mask, device, args.pred_len, "Baseline validation")
baseline_test = compute_with_global_mask(model, test_loader, test_regime_df, mask_controller, baseline_mask, device, args.pred_len, "Baseline test")

print("Baseline val:", baseline_val["overall"], "| test:", baseline_test["overall"])

Baseline validation:   0%|          | 0/358 [00:00<?, ?it/s]

Baseline test:   0%|          | 0/358 [00:00<?, ?it/s]

Baseline val: {'overall_mse': 0.37885430447871365, 'overall_mae': 0.41427090555364243} | test: {'overall_mse': 0.2876100955825275, 'overall_mae': 0.34491427588515}


## 13. Head importance (leave-one-out), on a validation subsample

Same method as `03_head_importance.ipynb`: mask one head at a time, measure the change in MSE overall and per regime. To keep runtime bounded on ETTm1's larger validation set, this uses a fixed random subsample (`HEAD_IMPORTANCE_SUBSAMPLE_SIZE=1500` windows, stratified implicitly by taking a uniform random sample across all validation windows) rather than the full set.

In [16]:
HEAD_IMPORTANCE_SUBSAMPLE_SIZE = 1500
IMPORTANCE_SEED = 7

rng = random.Random(IMPORTANCE_SEED)
all_val_indices = list(range(len(regime_df)))
if len(all_val_indices) > HEAD_IMPORTANCE_SUBSAMPLE_SIZE:
    importance_indices = sorted(rng.sample(all_val_indices, HEAD_IMPORTANCE_SUBSAMPLE_SIZE))
else:
    importance_indices = all_val_indices

from torch.utils.data import Subset, DataLoader
importance_subset = Subset(vali_data, importance_indices)
importance_loader = DataLoader(importance_subset, batch_size=32, shuffle=False)
importance_regime_df = regime_df.iloc[importance_indices].reset_index(drop=True)

print("Head importance will use", len(importance_indices), "validation windows.")

baseline_importance_ref = compute_with_global_mask(
    model, importance_loader, importance_regime_df, mask_controller, baseline_mask, device, args.pred_len, "Baseline (importance subsample)"
)
baseline_importance_overall = baseline_importance_ref["overall"]["overall_mse"]
baseline_importance_by_regime = {
    row["regime"]: row["mse"] for _, row in baseline_importance_ref["regime_summary"].iterrows()
}

importance_records = []
for layer_idx in range(num_layers):
    for head_idx in range(num_heads):
        mask = baseline_mask.clone()
        mask[layer_idx, head_idx] = 0.0

        masked_result = compute_with_global_mask(
            model, importance_loader, importance_regime_df, mask_controller, mask, device, args.pred_len,
            desc=f"L{layer_idx}H{head_idx}",
        )
        masked_overall = masked_result["overall"]["overall_mse"]
        masked_by_regime = {row["regime"]: row["mse"] for _, row in masked_result["regime_summary"].iterrows()}

        record = {
            "layer": layer_idx,
            "head": head_idx,
            "overall_importance": masked_overall - baseline_importance_overall,
        }
        for regime in ["trend", "seasonal", "residual"]:
            base_r = baseline_importance_by_regime.get(regime, np.nan)
            masked_r = masked_by_regime.get(regime, np.nan)
            record[f"{regime}_importance"] = masked_r - base_r if not np.isnan(base_r) else np.nan

        importance_records.append(record)

importance_df = pd.DataFrame(importance_records)
importance_df.to_csv(ETTM1_RESULTS_DIR / "ettm1_head_importance.csv", index=False)
display(importance_df)

Head importance will use 1500 validation windows.


Baseline (importance subsample):   0%|          | 0/47 [00:00<?, ?it/s]

L0H0:   0%|          | 0/47 [00:00<?, ?it/s]

L0H1:   0%|          | 0/47 [00:00<?, ?it/s]

L0H2:   0%|          | 0/47 [00:00<?, ?it/s]

...

L2H4:   0%|          | 0/47 [00:00<?, ?it/s]

L2H5:   0%|          | 0/47 [00:00<?, ?it/s]

L2H6:   0%|          | 0/47 [00:00<?, ?it/s]

L2H7:   0%|          | 0/47 [00:00<?, ?it/s]

,layer,head,overall_importance,trend_importance,seasonal_importance,residual_importance
0,0,0,0.004985,0.005008,0.005916,0.002019
1,0,1,0.001218,0.003929,0.000096,-0.000084
2,0,2,-0.009167,-0.005073,-0.012288,-0.006644
3,0,3,0.007750,0.008033,0.006148,0.012290
4,0,4,0.013830,0.007416,0.014858,0.022028
5,0,5,0.011656,0.013015,0.013034,0.004898
6,0,6,0.014747,0.015824,0.014366,0.014026
7,0,7,-0.008662,-0.008532,-0.009398,-0.006575
8,1,0,-0.002013,-0.002698,-0.001766,-0.001569
9,1,1,-0.002023,-0.000283,-0.003256,-0.001245


## 14. Static 25% pruning mask (lowest overall importance)

In [17]:
static_prune_df = importance_df.sort_values("overall_importance").head(6).copy()
print("Static pruning candidates (ETTm1):")
display(static_prune_df[["layer", "head", "overall_importance"]])

static_mask = baseline_mask.clone()
for _, row in static_prune_df.iterrows():
    static_mask[int(row["layer"]), int(row["head"])] = 0.0

static_val = compute_with_global_mask(model, vali_loader, regime_df, mask_controller, static_mask, device, args.pred_len, "Static 25% validation")
static_test = compute_with_global_mask(model, test_loader, test_regime_df, mask_controller, static_mask, device, args.pred_len, "Static 25% test")

print("Static val:", static_val["overall"], "| test:", static_test["overall"])

Static pruning candidates (ETTm1):


,layer,head,overall_importance
2,0,2,-0.009167
7,0,7,-0.008662
17,2,1,-0.007066
16,2,0,-0.006862
13,1,5,-0.004693
23,2,7,-0.003713


Static 25% validation:   0%|          | 0/358 [00:00<?, ?it/s]

Static 25% test:   0%|          | 0/358 [00:00<?, ?it/s]

Static val: {'overall_mse': 0.360656621022154, 'overall_mae': 0.39486975364533766} | test: {'overall_mse': 0.29797178286333575, 'overall_mae': 0.34879192249388874}


## 15. Joint (greedy) dynamic 75% keep, per regime

Same greedy backward elimination as notebook 10: per regime, remove the single head that least harms masked MSE, one at a time, until 18/24 heads remain, using a random subsample (capped at 256 windows) of that regime's validation windows for speed.

In [18]:
GREEDY_SUBSAMPLE_SIZE = 256
GREEDY_SEED = 42
KEEP_RATIO = 0.75
TARGET_KEEP = round(total_heads * KEEP_RATIO)

def collect_batches(dataset, indices, device, batch_size=32):
    subset = Subset(dataset, indices)
    loader = DataLoader(subset, batch_size=batch_size, shuffle=False)
    batches = []
    for batch in loader:
        bx, by, bxm, bym = [t.float().to(device) for t in batch]
        batches.append((bx, bxm, by, bym))
    return batches


def eval_masked_mse_on_batches(model, mask_controller, mask, batches, pred_len):
    model.eval()
    mask_controller.set_mask(mask)
    total_se, total_count = 0.0, 0
    with torch.no_grad():
        for bx, bxm, by, bym in batches:
            outputs = model(bx, bxm, by, bym)
            true = by[:, -pred_len:, :]
            total_se += mse_criterion(outputs, true).mean(dim=(1, 2)).sum().item()
            total_count += bx.shape[0]
    return total_se / total_count


greedy_rng = random.Random(GREEDY_SEED)
regime_batch_caches = {}
for regime in ["trend", "seasonal", "residual"]:
    indices = regime_df.index[regime_df["regime"] == regime].tolist()
    if len(indices) > GREEDY_SUBSAMPLE_SIZE:
        indices = sorted(greedy_rng.sample(indices, GREEDY_SUBSAMPLE_SIZE))
    regime_batch_caches[regime] = collect_batches(vali_data, indices, device)
    print(f"{regime}: greedy search will use {len(indices)} validation windows")

trend: greedy search will use 256 validation windows
seasonal: greedy search will use 256 validation windows
residual: greedy search will use 256 validation windows


In [19]:
def greedy_backward_eliminate(model, mask_controller, batches, num_layers, num_heads, target_keep, pred_len, desc=""):
    all_heads = [(l, h) for l in range(num_layers) for h in range(num_heads)]
    active = set(all_heads)
    removal_order = []

    while len(active) > target_keep:
        best_head, best_mse = None, None
        for candidate in active:
            trial_active = active - {candidate}
            mask = torch.zeros(num_layers, num_heads, dtype=torch.float32)
            for (l, h) in trial_active:
                mask[l, h] = 1.0
            mse = eval_masked_mse_on_batches(model, mask_controller, mask, batches, pred_len)
            if best_mse is None or mse < best_mse:
                best_mse, best_head = mse, candidate

        active.discard(best_head)
        removal_order.append({"step": len(removal_order) + 1, "layer": best_head[0], "head": best_head[1],
                               "resulting_mse": best_mse, "remaining_heads": len(active)})
        print(f"[{desc}] step {len(removal_order)}: removed L{best_head[0]}H{best_head[1]} -> mse={best_mse:.6f} ({len(active)} remain)")

    final_mask = torch.zeros(num_layers, num_heads, dtype=torch.float32)
    for (l, h) in active:
        final_mask[l, h] = 1.0
    return final_mask, pd.DataFrame(removal_order)


regime_masks_joint = {}
removal_order_dfs = {}
for regime in ["trend", "seasonal", "residual"]:
    final_mask, removal_df = greedy_backward_eliminate(
        model, mask_controller, regime_batch_caches[regime], num_layers, num_heads, TARGET_KEEP, args.pred_len, desc=regime
    )
    regime_masks_joint[regime] = final_mask
    removal_order_dfs[regime] = removal_df
    print(f"{regime} final active heads: {int(final_mask.sum().item())}")

[trend] step 1: removed L0H7 -> mse=0.355315 (23 remain)
[trend] step 2: removed L2H0 -> mse=0.349746 (22 remain)
[trend] step 3: removed L0H2 -> mse=0.345255 (21 remain)
[trend] step 4: removed L1H2 -> mse=0.342777 (20 remain)
[trend] step 5: removed L2H3 -> mse=0.342638 (19 remain)
[trend] step 6: removed L1H3 -> mse=0.342843 (18 remain)
trend final active heads: 18
[seasonal] step 1: removed L0H2 -> mse=0.385454 (23 remain)
[seasonal] step 2: removed L2H0 -> mse=0.373884 (22 remain)
[seasonal] step 3: removed L0H7 -> mse=0.367878 (21 remain)
[seasonal] step 4: removed L1H5 -> mse=0.366149 (20 remain)
[seasonal] step 5: removed L2H3 -> mse=0.365633 (19 remain)
[seasonal] step 6: removed L2H7 -> mse=0.366444 (18 remain)
seasonal final active heads: 18
[residual] step 1: removed L0H2 -> mse=0.289904 (23 remain)
[residual] step 2: removed L1H5 -> mse=0.284952 (22 remain)
[residual] step 3: removed L0H7 -> mse=0.280224 (21 remain)
[residual] step 4: removed L2H7 -> mse=0.277643 (20 remai

## 16. Evaluate the joint dynamic masks on the full validation and test sets

In [20]:
joint_val = compute_dynamic(model, vali_loader, regime_df, regime_masks_joint, mask_controller, device, args.pred_len, "Joint dynamic validation")
joint_test = compute_dynamic(model, test_loader, test_regime_df, regime_masks_joint, mask_controller, device, args.pred_len, "Joint dynamic test")

print("Joint dynamic val:", joint_val["overall"], "| test:", joint_test["overall"])

Joint dynamic validation:   0%|          | 0/358 [00:00<?, ?it/s]

Joint dynamic test:   0%|          | 0/358 [00:00<?, ?it/s]

Joint dynamic val: {'overall_mse': 0.35815327705778754, 'overall_mae': 0.395860837864928} | test: {'overall_mse': 0.28890853404118405, 'overall_mae': 0.34503171998901494}


## 17. Consolidated comparison and save results

In [21]:
summary_df = pd.DataFrame([
    {"setting": "ETTm1_no_pruning", "val_mse": baseline_val["overall"]["overall_mse"], "val_mae": baseline_val["overall"]["overall_mae"],
     "test_mse": baseline_test["overall"]["overall_mse"], "test_mae": baseline_test["overall"]["overall_mae"]},
    {"setting": "ETTm1_static_pruning_25", "val_mse": static_val["overall"]["overall_mse"], "val_mae": static_val["overall"]["overall_mae"],
     "test_mse": static_test["overall"]["overall_mse"], "test_mae": static_test["overall"]["overall_mae"]},
    {"setting": "ETTm1_dynamic_75_joint", "val_mse": joint_val["overall"]["overall_mse"], "val_mae": joint_val["overall"]["overall_mae"],
     "test_mse": joint_test["overall"]["overall_mse"], "test_mae": joint_test["overall"]["overall_mae"]},
])

baseline_test_mse = summary_df.loc[summary_df["setting"] == "ETTm1_no_pruning", "test_mse"].iloc[0]
summary_df["test_mse_change_percent"] = (summary_df["test_mse"] - baseline_test_mse) / baseline_test_mse * 100

display(summary_df)

summary_df.to_csv(ETTM1_RESULTS_DIR / "ettm1_consolidated_comparison.csv", index=False)
static_prune_df.to_csv(ETTM1_RESULTS_DIR / "ettm1_static_prune_heads.csv", index=False)

for regime in ["trend", "seasonal", "residual"]:
    removal_order_dfs[regime].to_csv(ETTM1_RESULTS_DIR / f"ettm1_{regime}_greedy_removal_order.csv", index=False)
    mask_df = pd.DataFrame(
        regime_masks_joint[regime].numpy(),
        index=[f"layer_{i}" for i in range(num_layers)],
        columns=[f"head_{j}" for j in range(num_heads)],
    )
    mask_df.to_csv(ETTM1_RESULTS_DIR / f"ettm1_{regime}_joint_dynamic_keep_75_mask.csv")

joint_val["regime_summary"].to_csv(ETTM1_RESULTS_DIR / "ettm1_joint_validation_regime_summary.csv", index=False)
joint_test["regime_summary"].to_csv(ETTM1_RESULTS_DIR / "ettm1_joint_test_regime_summary.csv", index=False)

print("\nSaved all ETTm1 extension outputs to:", ETTM1_RESULTS_DIR)
for path in sorted(ETTM1_RESULTS_DIR.iterdir()):
    print(" -", path.name)

if summary_df.loc[summary_df['setting'] == 'ETTm1_dynamic_75_joint', 'test_mse_change_percent'].iloc[0] < \
   summary_df.loc[summary_df['setting'] == 'ETTm1_static_pruning_25', 'test_mse_change_percent'].iloc[0]:
    print("\nJoint dynamic selection outperforms static pruning on ETTm1 too, consistent with the ETTh1 finding.")
else:
    print("\nOn ETTm1, static pruning outperformed joint dynamic selection - the ETTh1 finding does not fully transfer.")

,setting,val_mse,val_mae,test_mse,test_mae,test_mse_change_percent
0,ETTm1_no_pruning,0.378854,0.414271,0.287610,0.344914,0.000000
1,ETTm1_static_pruning_25,0.360657,0.394870,0.297972,0.348792,3.602686
2,ETTm1_dynamic_75_joint,0.358153,0.395861,0.288909,0.345032,0.451458



Saved all ETTm1 extension outputs to: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/ettm1_extension/results
 - ettm1_consolidated_comparison.csv
 - ettm1_head_importance.csv
 - ettm1_joint_test_regime_summary.csv
 - ettm1_joint_validation_regime_summary.csv
 - ettm1_residual_greedy_removal_order.csv
 - ettm1_residual_joint_dynamic_keep_75_mask.csv
 - ettm1_seasonal_greedy_removal_order.csv
 - ettm1_seasonal_joint_dynamic_keep_75_mask.csv
 - ettm1_static_prune_heads.csv
 - ettm1_trend_greedy_removal_order.csv
 - ettm1_trend_joint_dynamic_keep_75_mask.csv

Joint dynamic selection outperforms static pruning on ETTm1 too, consistent with the ETTh1 finding.
